# AirOps 360 - Open-Meteo Bronze Ingestion Pilot

**Task:** Week 3 Task 26  
**Scope:** April 2026, ORD + ATL only  
**Source:** Open-Meteo Historical Weather API  
**Target:** `brz_weather_api_raw`  
**Audit:** `brz_ingestion_audit`

## Pilot purpose

Prove the configured API-to-OneLake-to-Bronze ingestion pattern without
performing a full 15-airport or multi-month backfill.

Bronze preserves one raw API response per airport/month together with
request, airport, time-zone, and execution lineage metadata. Hourly weather
normalization is intentionally deferred to Silver.

In [1]:
import json
import uuid
import hashlib
import time
import random
from datetime import datetime, timezone

import requests

from pyspark.sql import functions as F
from delta.tables import DeltaTable


# ---------------------------------------------------------
# TASK 26 CONFIGURATION
# ---------------------------------------------------------

YEAR = 2026
MONTH = 4

START_DATE = "2026-04-01"
END_DATE = "2026-04-30"

CONTRACT_VERSION = "0.1"
VARIABLE_SET_VERSION = "wx_v1"

SOURCE_NAME = "open_meteo_historical_weather"

API_URL = "https://archive-api.open-meteo.com/v1/archive"

WEATHER_TABLE = "brz_weather_api_raw"
AUDIT_TABLE = "brz_ingestion_audit"

AIRPORT_CONFIG_PATH = "Files/reference/airports/airports_v0.1.csv"

PILOT_AIRPORTS = ["ORD", "ATL"]

HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "snowfall",
    "weather_code",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
]

RUN_ID = str(uuid.uuid4())

print("TASK 26 CONFIGURATION")
print("---------------------")
print("Run ID       :", RUN_ID)
print("Period       :", START_DATE, "to", END_DATE)
print("Pilot airports:", PILOT_AIRPORTS)
print("Variables    :", HOURLY_VARIABLES)

StatementMeta(, f4beca5a-8f71-4a06-b4a3-1d3caf754b9d, 3, Finished, Available, Finished, False)

TASK 26 CONFIGURATION
---------------------
Run ID       : 29f58e2a-7271-46eb-ae1a-69f108905c5b
Period       : 2026-04-01 to 2026-04-30
Pilot airports: ['ORD', 'ATL']
Variables    : ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'snowfall', 'weather_code', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m']


In [2]:
airport_config = (
    spark.read
    .option("header", True)
    .csv(AIRPORT_CONFIG_PATH)
)

pilot_config = (
    airport_config
    .filter(
        (F.lower(F.col("active")) == "true")
        & F.col("iata_code").isin(PILOT_AIRPORTS)
    )
    .select(
        F.col("rank").cast("int").alias("rank"),
        "iata_code",
        "airport_name",
        F.col("latitude").cast("double").alias("latitude"),
        F.col("longitude").cast("double").alias("longitude"),
        "iana_timezone",
    )
    .orderBy("rank")
)

display(pilot_config)

assert pilot_config.count() == 2, "Expected exactly ORD and ATL."

codes = {r["iata_code"] for r in pilot_config.collect()}

assert codes == {"ORD", "ATL"}, (
    f"Unexpected airport selection: {codes}"
)

print("AIRPORT CONFIG VALIDATION: PASS")

StatementMeta(, f4beca5a-8f71-4a06-b4a3-1d3caf754b9d, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 51ffc4c8-a574-4e47-9c0d-67f8ac82b8fb)

AIRPORT CONFIG VALIDATION: PASS


In [3]:
print("WEATHER BRONZE SCHEMA")
spark.table(WEATHER_TABLE).printSchema()

print("\nAUDIT TABLE SCHEMA")
spark.table(AUDIT_TABLE).printSchema()

print(
    "Existing weather Bronze rows:",
    spark.table(WEATHER_TABLE).count()
)

StatementMeta(, f4beca5a-8f71-4a06-b4a3-1d3caf754b9d, 5, Finished, Available, Finished, False)

WEATHER BRONZE SCHEMA
root
 |-- response_json: string (nullable = true)
 |-- airport_code: string (nullable = true)
 |-- request_start_date: date (nullable = true)
 |-- request_end_date: date (nullable = true)
 |-- response_timezone: string (nullable = true)
 |-- response_utc_offset_seconds: integer (nullable = true)
 |-- _bronze_run_id: string (nullable = true)
 |-- _bronze_load_id: string (nullable = true)
 |-- _bronze_batch_key: string (nullable = true)
 |-- _bronze_contract_version: string (nullable = true)
 |-- _bronze_source_name: string (nullable = true)
 |-- _bronze_ingested_at_utc: timestamp (nullable = true)


AUDIT TABLE SCHEMA
root
 |-- run_id: string (nullable = true)
 |-- load_id: string (nullable = true)
 |-- attempt_no: integer (nullable = true)
 |-- batch_key: string (nullable = true)
 |-- contract_version: string (nullable = true)
 |-- source_name: string (nullable = true)
 |-- source_object: string (nullable = true)
 |-- source_uri: string (nullable = true)
 |-- sour

In [4]:
def align_to_target_table(df, table_name):
    """
    Cast/reorder an incoming DataFrame so it exactly matches
    the already-governed Delta table schema.
    """
    target_schema = spark.table(table_name).schema
    target_columns = [f.name for f in target_schema.fields]

    missing = [
        name for name in target_columns
        if name not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{table_name}: missing required columns {missing}"
        )

    return df.select(
        *[
            F.col(field.name)
            .cast(field.dataType.simpleString())
            .alias(field.name)
            for field in target_schema.fields
        ]
    )


def request_open_meteo(params, max_attempts=3):
    """
    Retry only transient failures:
      - request/network exception
      - HTTP 429
      - HTTP 5xx

    Invalid 4xx requests fail immediately.
    """

    last_error = None

    for attempt in range(1, max_attempts + 1):

        try:
            response = requests.get(
                API_URL,
                params=params,
                timeout=60,
            )

            if response.status_code == 200:
                return response, attempt

            if (
                response.status_code == 429
                or 500 <= response.status_code <= 599
            ):
                last_error = RuntimeError(
                    f"Retryable HTTP {response.status_code}"
                )

            else:
                raise RuntimeError(
                    f"Non-retryable HTTP "
                    f"{response.status_code}: "
                    f"{response.text[:500]}"
                )

        except requests.RequestException as exc:
            last_error = exc

        if attempt < max_attempts:
            delay = (2 ** (attempt - 1)) + random.uniform(0, 0.5)

            print(
                f"Transient failure. "
                f"Retrying in {delay:.2f}s..."
            )

            time.sleep(delay)

    raise RuntimeError(
        f"Open-Meteo request failed after "
        f"{max_attempts} attempts: {last_error}"
    )

StatementMeta(, f4beca5a-8f71-4a06-b4a3-1d3caf754b9d, 6, Finished, Available, Finished, False)

In [5]:
weather_rows = []
load_evidence = []

EXPECTED_HOURS = 30 * 24   # April = 30 days

for airport in pilot_config.collect():

    code = airport["iata_code"]
    latitude = airport["latitude"]
    longitude = airport["longitude"]
    airport_timezone = airport["iana_timezone"]

    load_id = str(uuid.uuid4())

    batch_key = (
        f"{SOURCE_NAME}|"
        f"{code}|"
        f"{YEAR}|"
        f"{MONTH:02d}|"
        f"{VARIABLE_SET_VERSION}"
    )

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": ",".join(HOURLY_VARIABLES),
        "timezone": airport_timezone,
    }

    print(f"\nRequesting {code} ...")

    response, attempt_no = request_open_meteo(params)

    response_json_text = response.text
    payload = response.json()

    hourly_times = payload.get("hourly", {}).get("time", [])

    if len(hourly_times) != EXPECTED_HOURS:
        raise ValueError(
            f"{code}: expected {EXPECTED_HOURS} hourly "
            f"observations but received {len(hourly_times)}"
        )

    source_hash = hashlib.sha256(
        response.content
    ).hexdigest()

    response_timezone = payload.get("timezone")
    utc_offset_seconds = payload.get(
        "utc_offset_seconds"
    )

    if response_timezone != airport_timezone:
        raise ValueError(
            f"{code}: requested timezone "
            f"{airport_timezone}, response returned "
            f"{response_timezone}"
        )

    raw_directory = (
        f"Files/raw/weather/"
        f"airport={code}/"
        f"year={YEAR}/"
        f"month={MONTH:02d}"
    )

    raw_filename = (
        f"open_meteo_{code}_{YEAR}_{MONTH:02d}.json"
    )

    raw_path = f"{raw_directory}/{raw_filename}"

    notebookutils.fs.mkdirs(raw_directory)

    notebookutils.fs.put(
        raw_path,
        response_json_text,
        True,
    )

    target_before = (
        spark.table(WEATHER_TABLE)
        .filter(
            F.col("_bronze_batch_key") == batch_key
        )
        .count()
    )

    ingested_at = datetime.now(timezone.utc)

    weather_rows.append(
        {
            "response_json": response_json_text,
            "airport_code": code,
            "request_start_date": START_DATE,
            "request_end_date": END_DATE,
            "response_timezone": response_timezone,
            "response_utc_offset_seconds": utc_offset_seconds,
            "_bronze_run_id": RUN_ID,
            "_bronze_load_id": load_id,
            "_bronze_batch_key": batch_key,
            "_bronze_contract_version": CONTRACT_VERSION,
            "_bronze_source_name": SOURCE_NAME,
            "_bronze_ingested_at_utc": ingested_at,
        }
    )

    load_evidence.append(
        {
            "airport_code": code,
            "load_id": load_id,
            "batch_key": batch_key,
            "attempt_no": attempt_no,
            "source_hash": source_hash,
            "source_uri": response.url,
            "raw_filename": raw_filename,
            "raw_path": raw_path,
            "hourly_count": len(hourly_times),
            "source_column_count": len(
                payload.get("hourly", {}).keys()
            ),
            "target_before": target_before,
            "parameters_json": json.dumps(
                params,
                sort_keys=True,
            ),
        }
    )

    print(
        f"{code}: "
        f"{len(hourly_times)} hourly observations | "
        f"hash={source_hash[:12]}... | "
        f"attempt={attempt_no}"
    )


print("\nOPEN-METEO API EXTRACTION: PASS")

StatementMeta(, f4beca5a-8f71-4a06-b4a3-1d3caf754b9d, 7, Finished, Available, Finished, False)


Requesting ORD ...
ORD: 720 hourly observations | hash=c75a37281aae... | attempt=1

Requesting ATL ...
ATL: 720 hourly observations | hash=8842f586979f... | attempt=1

OPEN-METEO API EXTRACTION: PASS


In [6]:
incoming_weather = spark.createDataFrame(weather_rows)

incoming_weather = align_to_target_table(
    incoming_weather,
    WEATHER_TABLE,
)

display(
    incoming_weather.select(
        "airport_code",
        "request_start_date",
        "request_end_date",
        "response_timezone",
        "_bronze_run_id",
        "_bronze_load_id",
        "_bronze_batch_key",
    )
)

weather_delta = DeltaTable.forName(
    spark,
    WEATHER_TABLE,
)

(
    weather_delta.alias("target")
    .merge(
        incoming_weather.alias("source"),
        """
        target._bronze_batch_key
        = source._bronze_batch_key
        """,
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("BRONZE WEATHER MERGE: PASS")

StatementMeta(, f4beca5a-8f71-4a06-b4a3-1d3caf754b9d, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 24989ca8-c86f-4fc7-bfd5-c8e090be24f9)

BRONZE WEATHER MERGE: PASS


In [7]:
audit_rows = []

for item in load_evidence:

    batch_key = item["batch_key"]

    target_after = (
        spark.table(WEATHER_TABLE)
        .filter(
            F.col("_bronze_batch_key") == batch_key
        )
        .count()
    )

    reconciliation_status = (
        "PASS"
        if (
            item["hourly_count"] == EXPECTED_HOURS
            and target_after == 1
        )
        else "FAIL"
    )

    audit_rows.append(
        {
            "run_id": RUN_ID,
            "load_id": item["load_id"],
            "attempt_no": item["attempt_no"],
            "batch_key": batch_key,
            "contract_version": CONTRACT_VERSION,
            "source_name": SOURCE_NAME,
            "source_object": item["raw_filename"],
            "source_uri": item["source_uri"],
            "source_file_name": item["raw_filename"],
            "source_hash": item["source_hash"],
            "load_year": YEAR,
            "load_month": MONTH,
            "airport_code": item["airport_code"],
            "variable_set_version": VARIABLE_SET_VERSION,
            "pipeline_name": "manual_week3_pilot",
            "activity_name": "nb_bronze_ingest_weather",
            "load_mode": "historical",
            "started_at_utc": datetime.now(timezone.utc),
            "completed_at_utc": datetime.now(timezone.utc),
            "ingestion_status": "SUCCEEDED",
            "source_row_count": item["hourly_count"],
            "source_column_count": item["source_column_count"],
            "processed_row_count": 1,
            "target_row_count_before": item["target_before"],
            "target_row_count_after": target_after,
            "rejected_row_count": 0,
            "reconciliation_status": reconciliation_status,
            "parameters_json": item["parameters_json"],
            "error_class": "",
            "error_message": "",
        }
    )


incoming_audit = spark.createDataFrame(audit_rows)

incoming_audit = align_to_target_table(
    incoming_audit,
    AUDIT_TABLE,
)

(
    incoming_audit.write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)

print("INGESTION AUDIT APPEND: PASS")

StatementMeta(, f4beca5a-8f71-4a06-b4a3-1d3caf754b9d, 9, Finished, Available, Finished, False)

INGESTION AUDIT APPEND: PASS


In [8]:
pilot_batch_keys = [
    item["batch_key"]
    for item in load_evidence
]

pilot_bronze = (
    spark.table(WEATHER_TABLE)
    .filter(
        F.col("_bronze_batch_key")
        .isin(pilot_batch_keys)
    )
)

pilot_row_count = pilot_bronze.count()

batch_count = (
    pilot_bronze
    .select("_bronze_batch_key")
    .distinct()
    .count()
)

run_count = (
    pilot_bronze
    .select("_bronze_run_id")
    .distinct()
    .count()
)

load_count = (
    pilot_bronze
    .select("_bronze_load_id")
    .distinct()
    .count()
)

required_metadata = [
    "airport_code",
    "request_start_date",
    "request_end_date",
    "response_timezone",
    "response_utc_offset_seconds",
    "_bronze_run_id",
    "_bronze_load_id",
    "_bronze_batch_key",
    "_bronze_contract_version",
    "_bronze_source_name",
    "_bronze_ingested_at_utc",
]

metadata_null_count = 0

for column_name in required_metadata:
    metadata_null_count += (
        pilot_bronze
        .filter(F.col(column_name).isNull())
        .count()
    )


assert pilot_row_count == 2, (
    f"Expected 2 Bronze pilot rows, got "
    f"{pilot_row_count}"
)

assert batch_count == 2, (
    f"Expected 2 distinct batches, got "
    f"{batch_count}"
)

assert run_count == 1, (
    f"Expected one shared run_id, got "
    f"{run_count}"
)

assert load_count == 2, (
    f"Expected two load_ids, got "
    f"{load_count}"
)

assert metadata_null_count == 0, (
    f"Metadata null count = "
    f"{metadata_null_count}"
)

for item in load_evidence:

    assert item["hourly_count"] == 720

    assert notebookutils.fs.exists(
        item["raw_path"]
    )


print("==========================================")
print("TASK 26 — OPEN-METEO BRONZE PILOT")
print("==========================================")
print("Pilot airports          : ORD, ATL")
print("Period                  : 2026-04-01 to 2026-04-30")
print("Hourly observations ORD : 720")
print("Hourly observations ATL : 720")
print("Bronze response rows    :", pilot_row_count)
print("Distinct batch keys     :", batch_count)
print("Distinct run IDs        :", run_count)
print("Distinct load IDs       :", load_count)
print("Metadata null count     :", metadata_null_count)
print("Raw files landed        : 2")
print("==========================================")
print("TASK 26 STATUS: PASS")

StatementMeta(, f4beca5a-8f71-4a06-b4a3-1d3caf754b9d, 10, Finished, Available, Finished, False)

TASK 26 — OPEN-METEO BRONZE PILOT
Pilot airports          : ORD, ATL
Period                  : 2026-04-01 to 2026-04-30
Hourly observations ORD : 720
Hourly observations ATL : 720
Bronze response rows    : 2
Distinct batch keys     : 2
Distinct run IDs        : 1
Distinct load IDs       : 2
Metadata null count     : 0
Raw files landed        : 2
TASK 26 STATUS: PASS


# Task 26 - API-to-Bronze Pattern

## Pilot scope
- Source: Open-Meteo Historical Weather API
- Period: April 2026
- Airports: ORD and ATL
- Configuration source: `airports_v0.1.csv`
- Variable set: `wx_v1`

## Pattern

`governed airport config`
→ `parameterized REST request`
→ `retry transient API failures`
→ `validate response/time range`
→ `preserve raw JSON in OneLake`
→ `publish one raw response row per airport/month`
→ `attach batch/run/load lineage`
→ `append operational audit evidence`
→ `normalize hourly observations later in Silver`

## Bronze grain

`brz_weather_api_raw` = one airport/month Open-Meteo response.

The pilot intentionally does not flatten hourly weather values in Bronze.
Source structure is preserved so ingestion and analytical transformation
remain separate concerns.

## Identities

- One `run_id` for the pilot execution.
- Separate `load_id` for each airport API request.
- Stable deterministic `batch_key` per airport/month/variable-set version.

Example:

`open_meteo_historical_weather|ORD|2026|04|wx_v1`

## Reconciliation

- ORD hourly observations: 720
- ATL hourly observations: 720
- Expected Bronze response rows: 2
- Expected logical batches: 2
- Expected metadata null count: 0
